This pipeline test uses:
- **Count and area:** OpenCV
- **Stage Classificator:** CNN
- **Action Motor:** Strict Rules (Python If/Else)

In [ ]:
from pathlib import Path
import sys

In [ ]:
project_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "utils" / "utls_install_libs.py").is_file()),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not find utils/utls_install_libs.py")

sys.path.insert(0, str(project_root))

In [ ]:
from utils.utls_install_libs import install_libs

In [ ]:
bibliotecas_necessarias = [
    'google-api-python-client',
    'google-auth-httplib2',
    'google-auth-oauthlib',
    'pillow',
    'numpy',
    'opencv-python',
    'tensorflow',
    'matplotlib',
    'sklearn',
    'pandas'
]

In [ ]:
install_libs(bibliotecas_necessarias)

In [ ]:
import numpy as np
import pandas as pd
import cv2
import time

from sklearn.model_selection import train_test_split

from utils.utls_work_images import build_dataset
from utils.utls_ia_models import build_fast_cnn

In [ ]:
print("📥 Baixando matrizes do Google Drive (Isso pode levar alguns segundos)...")
X_exp, y_exp = build_dataset(limit_per_class_collected=50, folder_name_collected="expulsion")
X_inc, y_inc = build_dataset(limit_per_class_collected=50, folder_name_collected="incubation")

# Juntando e dividindo em treino e teste
X_completo = np.concatenate((X_exp, X_inc), axis=0)
y_completo = np.concatenate((y_exp, y_inc), axis=0)
X_train, X_test, y_train, y_test = train_test_split(X_completo, y_completo, test_size=0.2, random_state=42)

In [ ]:
print("🧠 Treinando a CNN Customizada rapidamente para o benchmark...")
modelo_reator = build_fast_cnn()
# Treino rápido de 5 épocas apenas para termos os pesos ajustados para o teste
modelo_reator.fit(X_train, y_train, epochs=5, verbose=0)

In [ ]:
def calcular_area_bolhas_cv2(img_array):
    # Converte o array de volta de 0.0-1.0 para 0-255 (exigência do OpenCV)
    img_uint8 = (img_array * 255).astype(np.uint8)
    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    
    # Filtro Canny: Extrai a geometria física mapeando contornos de contraste
    bordas = cv2.Canny(gray, 50, 150)
    contornos, _ = cv2.findContours(bordas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    quantidade_bolhas = len(contornos)
    area_vapor = sum(cv2.contourArea(c) for c in contornos)
    area_total = 224 * 224
    percentual_vapor = (area_vapor / area_total) * 100
    
    return quantidade_bolhas, round(percentual_vapor, 2)

In [ ]:
def motor_de_regras(classe_predita, percentual_vapor):
    # Classe 0 = Expulsão, Classe 1 = Incubação
    if classe_predita == 0:
        if percentual_vapor > 30:
            return "CRÍTICO: Expulsão severa. Acionar resfriamento de emergência máximo."
        return "ALERTA ALTO: Expulsão detectada. Aumentar fluxo da bomba."
    else:
        if percentual_vapor > 15:
            return "ALERTA MÉDIO: Incubação acelerada. Monitorar temperatura."
        return "NORMAL: Incubação inicial. Manter operação."

In [ ]:
resultados = []
print("⏱️ Iniciando cronometragem e processamento do Pipeline 01...")

for i in range(len(X_test)):
    img = X_test[i]
    inicio_timer = time.perf_counter() # Inicia o cronômetro do hardware
    
    # Estágio 1: Contagem OpenCV
    qtd_bolhas, pct_vapor = calcular_area_bolhas_cv2(img)
    
    # Estágio 2: Classificação IA
    img_batch = np.expand_dims(img, axis=0) 
    probabilidade = modelo_reator.predict(img_batch, verbose=0)[0][0]
    classe_ia = 1 if probabilidade > 0.5 else 0 
    
    # Estágio 3: Motor de Regras
    acao_sugerida = motor_de_regras(classe_ia, pct_vapor)
    
    # Para o cronômetro (calculando em milissegundos)
    fim_timer = time.perf_counter()
    latencia_ms = round((fim_timer - inicio_timer) * 1000, 2)
    
    resultados.append({
        "ID_Teste": i,
        "Real": "Incubação" if y_test[i] == 1 else "Expulsão",
        "Predicao_IA": "Incubação" if classe_ia == 1 else "Expulsão",
        "Qtd_Bolhas": qtd_bolhas,
        "Vapor_%": pct_vapor,
        "Acao_Sugerida": acao_sugerida,
        "Latencia_ms": latencia_ms
    })

In [ ]:
df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv('benchmark_pipeline_01.csv', index=False)

print(f"\n✅ Benchmark concluído! Arquivo 'benchmark_pipeline_01.csv' salvo no disco.")
print(f"📊 Latência Média do Pipeline 01: {df_resultados['Latencia_ms'].mean():.2f} ms")
display(df_resultados.head())